# Pairwise State Tomography — Tutorial

This notebook demonstrates **pairwise state tomography** as described in the paper
[Pairwise tomography networks for many-body quantum systems](https://arxiv.org/abs/1909.12814).

The idea is to reconstruct the two-qubit reduced density matrix for every
pair of qubits in an *N*-qubit state using a number of measurement circuits
that grows only as **O(log N)** — far fewer than the O(3^N) circuits needed
for full state tomography.

The workflow has three steps:

1. **Generate circuits** with `pairwise_state_tomography_circuits`.
2. **Run them yourself** on whatever backend you choose.
3. **Analyse the results** with `PairwiseStateTomographyFitter`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import DensityMatrix, partial_trace, state_fidelity
from qiskit_aer import AerSimulator

from pairwise_tomography import (
    pairwise_state_tomography_circuits,
    PairwiseStateTomographyFitter,
)
from pairwise_tomography.utils import concurrence
from pairwise_tomography.visualization import draw_entanglement_graph

First let's build a circuit that we want to do tomography on. The circuit must **not** contain any measurements; those are added by
`pairwise_state_tomography_circuits` automatically.

In [2]:
nq = 5
q = QuantumRegister(nq, name='q')
qc = QuantumCircuit(q)

# A circuit with a mix of local rotations and entangling gates
qc.h(0)
qc.x(1)
qc.h(2)
qc.cx(2,3)

qc.draw()

┌───┐     
q_0: ┤ H ├─────
     ├───┤     
q_1: ┤ X ├─────
     ├───┤     
q_2: ┤ H ├──■──
     └───┘┌─┴─┐
q_3: ─────┤ X ├
          └───┘
q_4: ──────────

Next we use `pairwise_state_tomography_circuits` to create the circuits required for the tomography. Each is
a copy of the original circuit with a different set of single-qubit rotation + measurement gates appended.

For our *N* = 5 qubit example, the number of circuits is
3 + 6 × ⌈log₃(5)⌉ = 3 + 6 × 2 = **15**,
covering all C(5, 2) = 10 qubit pairs.

In [3]:
tomo_circs = pairwise_state_tomography_circuits(qc, q)
print(f"{len(tomo_circs)} measurement circuits generated")
print("Circuit names (= per-qubit basis tuples):")
for c in tomo_circs:
    print(" ", c.name)

15 measurement circuits generated
Circuit names (= per-qubit basis tuples):
  ('X', 'X', 'X', 'X', 'X')
  ('Y', 'Y', 'Y', 'Y', 'Y')
  ('Z', 'Z', 'Z', 'Z', 'Z')
  ('X', 'Y', 'Z', 'X', 'Y')
  ('X', 'Z', 'Y', 'X', 'Z')
  ('Y', 'X', 'Z', 'Y', 'X')
  ('Y', 'Z', 'X', 'Y', 'Z')
  ('Z', 'X', 'Y', 'Z', 'X')
  ('Z', 'Y', 'X', 'Z', 'Y')
  ('X', 'X', 'X', 'Y', 'Y')
  ('X', 'X', 'X', 'Z', 'Z')
  ('Y', 'Y', 'Y', 'X', 'X')
  ('Y', 'Y', 'Y', 'Z', 'Z')
  ('Z', 'Z', 'Z', 'X', 'X')
  ('Z', 'Z', 'Z', 'Y', 'Y')


Each circuit name encodes the Pauli basis measured on every qubit.
For example `('X', 'Y', 'Z', 'X', 'Y')` means qubit 0 is measured in X,
qubit 1 in Y, and so on. The fitter uses these names to route each result
to the right qubit pair.

In [4]:
# Draw one example circuit to see the measurement rotations
tomo_circs[6].draw()

┌───┐┌─────┐┌───┐       ┌─┐           
 q_0: ┤ H ├┤ Sdg ├┤ H ├───────┤M├───────────
      ├───┤└─────┘└┬─┬┘       └╥┘           
 q_1: ┤ X ├────────┤M├─────────╫────────────
      ├───┤        └╥┘  ┌───┐  ║ ┌─┐        
 q_2: ┤ H ├───■─────╫───┤ H ├──╫─┤M├────────
      └───┘ ┌─┴─┐   ║  ┌┴───┴┐ ║ └╥┘┌───┐┌─┐
 q_3: ──────┤ X ├───╫──┤ Sdg ├─╫──╫─┤ H ├┤M├
       ┌─┐  └───┘   ║  └─────┘ ║  ║ └───┘└╥┘
 q_4: ─┤M├──────────╫──────────╫──╫───────╫─
       └╥┘          ║          ║  ║       ║ 
c0: 5/══╩═══════════╩══════════╩══╩═══════╩═
        4           1          0  2       3

Next we run the circuits. This can be done on any Qiskit-compatible backend — a local
simulator, `AerSimulator`, or real hardware via qiskit-ibm-runtime.
The package does not care how you run them; it only needs the `Result`
object back.

In [5]:
backend = AerSimulator()
shots = 8192

job = backend.run(transpile(tomo_circs, backend), shots=shots)
result = job.result()

### Density matrix reconstruction

The results are then passed into the fitter to perform the analysis required for the tomography.

Pass the result, the circuit list, and the measured qubits to
`PairwiseStateTomographyFitter`, then call `.fit()`.

The default method `'lstsq'` returns a density matrix as its output. It performs linear inversion of the Pauli
expectation values followed by projection onto the nearest physical
(positive-semidefinite, trace-1) density matrix.

In [6]:
fitter = PairwiseStateTomographyFitter(result, tomo_circs, q)
rho_pairs = fitter.fit()   # returns {(i, j): 4×4 ndarray}

In [7]:
print("Reconstructed pairs:", list(rho_pairs.keys()))
print()
print("rho(0, 1) =\n", np.round(rho_pairs[(0, 1)], 4))

Reconstructed pairs: [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]

rho(0, 1) =
 [[ 0.0027+0.j     -0.0024+0.0003j -0.0035+0.0023j -0.0029-0.0028j]
 [-0.0024-0.0003j  0.0022-0.j      0.0005-0.0003j -0.0006+0.0041j]
 [-0.0035-0.0023j  0.0005+0.0003j  0.4996+0.j      0.4927+0.0006j]
 [-0.0029+0.0028j -0.0006-0.0041j  0.4927-0.0006j  0.4955+0.j    ]]


To verify the results we can compute the exact reduced density matrices from the ideal statevector
and check the fidelity.

In [8]:
rho_exact = DensityMatrix(qc).data

print(f"{'Pair':>8}  {'Fidelity':>10}")
print("-" * 22)
for (i, j), rho_ij in rho_pairs.items():
    keep = [i, j]
    trace_out = [k for k in range(nq) if k not in keep]
    rho_ref = partial_trace(rho_exact, trace_out).data
    f = state_fidelity(rho_ij, rho_ref)
    print(f"  ({i}, {j})  {f:>10.4f}")

    Pair    Fidelity
----------------------
  (0, 1)      0.9902
  (0, 2)      0.9952
  (0, 3)      0.9906
  (0, 4)      0.9948
  (1, 2)      0.9997
  (1, 3)      0.9997
  (1, 4)      0.9939
  (2, 3)      0.9906
  (2, 4)      0.9997
  (3, 4)      0.9997


### Expectation value reconstruction

Passing `output='expectation'` skips density-matrix reconstruction entirely
and returns all 15 non-trivial two-qubit Pauli expectation values directly
from the measurement counts. No fitting is performed.

In [9]:
exp = fitter.fit(output='expectation')
# Integer keys -> single-qubit Pauli expectation values
# Tuple keys   -> two-qubit correlators (no identity terms)

First let's look at the single qubit expectation values, which should be easy to predict given the circuit.

In [10]:
for qi in range(nq):
    print(f"qubit {qi}")
    for basis, val in sorted(exp[qi].items()):
        print(f"  <{basis}> = {val:+.4f}")

qubit 0
  <X> = +1.0000
  <Y> = -0.0038
  <Z> = +0.0042
qubit 1
  <X> = -0.0077
  <Y> = -0.0060
  <Z> = -1.0000
qubit 2
  <X> = +0.0112
  <Y> = +0.0055
  <Z> = +0.0002
qubit 3
  <X> = -0.0044
  <Y> = +0.0021
  <Z> = +0.0002
qubit 4
  <X> = -0.0046
  <Y> = -0.0120
  <Z> = +1.0000


Qubits 0 and 1 are in the states $|+\rangle$ and $|1\rangle$ respectively. We therefore expect the `XZ` expectation value to be $-1$ for this pair. Are we right?

In [11]:
exp[(0, 1)][('X', 'Z')]

-1.0

Qubits 2 and 3 should also give us expectation values for an entangled pair.

In [12]:
ev = exp[(2, 3)]
print("Expectation values for pair (2, 3):")
for key, val in sorted(ev.items()):
    print(f"  <{key[0]}_{key[1]}> = {val:+.1f}")

Expectation values for pair (2, 3):
  <X_X> = +1.0
  <X_Y> = -0.0
  <X_Z> = +0.0
  <Y_X> = +0.0
  <Y_Y> = -1.0
  <Y_Z> = +0.0
  <Z_X> = -0.0
  <Z_Y> = -0.0
  <Z_Z> = +1.0


### Fitting only a subset of pairs

If you only care about specific pairs, pass `pairs_list` to avoid computing
the others.

In [13]:
nearest_neighbour_pairs = [(0, 1), (1, 2), (2, 3), (3, 4)]
nn_result = fitter.fit(pairs_list=nearest_neighbour_pairs)
print("Fitted pairs:", list(nn_result.keys()))

Fitted pairs: [(0, 1), (1, 2), (2, 3), (3, 4)]


### Measuring only a subset of qubits

You can pass a list of individual qubits (or a mix of registers and qubits)
instead of a whole register.

In [14]:
# Tomograph only qubits 0, 2, and 4
q = qc.qregs[0]
subset = [q[0], q[2], q[4]]

subset_circs = pairwise_state_tomography_circuits(qc, subset)
print(f"{len(subset_circs)} circuits for a 3-qubit subset")

subset_result = backend.run(transpile(subset_circs, backend), shots=shots).result()
subset_fitter = PairwiseStateTomographyFitter(subset_result, subset_circs, subset)
subset_rho = subset_fitter.fit()

print("Fitted pairs (indices into subset list):")
for k in subset_rho:
    i_circ = qc.find_bit(subset[k[0]]).index
    j_circ = qc.find_bit(subset[k[1]]).index
    print(f"  subset index {k}  →  circuit qubits ({i_circ}, {j_circ})")

9 circuits for a 3-qubit subset
Fitted pairs (indices into subset list):
  subset index (0, 1)  →  circuit qubits (0, 2)
  subset index (0, 2)  →  circuit qubits (0, 4)
  subset index (1, 2)  →  circuit qubits (2, 4)


### Bipartite graphs: 9 circuits for any N

For a bipartite graph, like the nearest-neighbour graph of a square lattice, it is possible to perform a tomography limited to neighbouring pairs that uses just 9 Pauli combinations for
every edge, regardless of N.

Just pass `pairs_list` to `pairwise_state_tomography_circuits`. If the pairs
form a bipartite graph it automatically emits 9 circuits; otherwise it falls
back to the full O(log N) scheme.

In [15]:
rows, cols = 3, 3
nq_lat = rows * cols

ql = QuantumRegister(nq_lat, name='l')
qc_lat = QuantumCircuit(ql)
psi_lat = np.random.randn(2**nq_lat) + 1j * np.random.randn(2**nq_lat)
psi_lat /= np.linalg.norm(psi_lat)
qc_lat.initialize(psi_lat, ql)

# Nearest-neighbour pairs (register index == fitter index since ql is ordered)
nn_pairs = []
for r in range(rows):
    for c in range(cols):
        idx = r * cols + c
        if c + 1 < cols:
            nn_pairs.append((idx, idx + 1))
        if r + 1 < rows:
            nn_pairs.append((idx, idx + cols))

# Checkerboard is bipartite -> auto-detects and emits 9 circuits
lat_circs = pairwise_state_tomography_circuits(qc_lat, ql, pairs_list=nn_pairs)
print(f"{len(lat_circs)} circuits for a {rows}×{cols} lattice ({nq_lat} qubits)")
print("Circuit names:", [c.name for c in lat_circs])

9 circuits for a 3×3 lattice (9 qubits)
Circuit names: ["('X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X')", "('X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y', 'X')", "('X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X')", "('Y', 'X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y')", "('Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y')", "('Y', 'Z', 'Y', 'Z', 'Y', 'Z', 'Y', 'Z', 'Y')", "('Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z')", "('Z', 'Y', 'Z', 'Y', 'Z', 'Y', 'Z', 'Y', 'Z')", "('Z', 'Z', 'Z', 'Z', 'Z', 'Z', 'Z', 'Z', 'Z')"]


In [16]:
lat_result = backend.run(transpile(lat_circs, backend), shots=shots).result()
lat_fitter = PairwiseStateTomographyFitter(lat_result, lat_circs, ql)

nn_rho = lat_fitter.fit(pairs_list=nn_pairs)
print(f"{len(nn_rho)} nearest-neighbour pairs fitted")

12 nearest-neighbour pairs fitted


In [17]:
rho_lat_exact = DensityMatrix(qc_lat).data

print(f"{'Pair':>10}  {'Fidelity':>10}")
print("-" * 24)
for (i, j), rho_ij in nn_rho.items():
    trace_out = [k for k in range(nq_lat) if k not in (i, j)]
    rho_ref = partial_trace(rho_lat_exact, trace_out).data
    f = state_fidelity(rho_ij, rho_ref)
    print(f"  ({i:>2}, {j:>2})      {f:>10.4f}")

      Pair    Fidelity
------------------------
  ( 0,  1)          0.9998
  ( 0,  3)          0.9998
  ( 1,  2)          0.9995
  ( 1,  4)          0.9998
  ( 2,  5)          0.9998
  ( 3,  4)          0.9995
  ( 3,  6)          0.9992
  ( 4,  5)          0.9993
  ( 4,  7)          0.9997
  ( 5,  8)          0.9997
  ( 6,  7)          0.9995
  ( 7,  8)          0.9995
